<a href="https://colab.research.google.com/github/VuKhang-ict/vnli-educorpus/blob/main/notebooks/01_prepare_hf_demo_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas scikit-learn huggingface_hub

In [2]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

WORKDIR = Path("/content/vnli_hf_demo")
META_DIR = WORKDIR / "metadata"
WORKDIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

print("Đã tạo thư mục làm việc:", WORKDIR)

Đã tạo thư mục làm việc: /content/vnli_hf_demo


In [3]:
from google.colab import files

uploaded = files.upload()
print("Các file đã upload:", list(uploaded.keys()))

Saving sources_tracker.csv to sources_tracker.csv
Saving schema_vnli_educorpus.csv to schema_vnli_educorpus.csv
Saving vnli_demo_30.csv to vnli_demo_30.csv
Các file đã upload: ['sources_tracker.csv', 'schema_vnli_educorpus.csv', 'vnli_demo_30.csv']


In [4]:
demo_file = "vnli_demo_30.csv"
schema_file = "schema_vnli_educorpus.csv"
sources_file = "sources_tracker.csv"

df = pd.read_csv(demo_file)
schema_df = pd.read_csv(schema_file)
sources_df = pd.read_csv(sources_file)

required_cols = [
    "id", "premise", "hypothesis", "label", "topic", "grade_level",
    "subject", "source_id", "license", "difficulty", "negation_flag",
    "annotator_id", "adjudication_note", "split"
]

missing_cols = [c for c in required_cols if c not in df.columns]

print("Số dòng / số cột:", df.shape)
print("Thiếu cột:", missing_cols)
print("\nPhân bố nhãn:")
print(df["label"].value_counts(dropna=False))
print("\n5 dòng đầu:")
display(df.head())

Số dòng / số cột: (30, 14)
Thiếu cột: []

Phân bố nhãn:
label
ENT    10
CON    10
NEU    10
Name: count, dtype: int64

5 dòng đầu:


,id,premise,hypothesis,label,topic,grade_level,subject,source_id,license,difficulty,negation_flag,annotator_id,adjudication_note,split
0,VNLI_DEMO_0001,Đoạn văn yêu cầu người viết nêu luận điểm chín...,Phần mở đầu có thể trình bày luận điểm chính.,ENT,nghi_luan,12,Ngu_van,SRC0001,author_owned,easy,no,author_v1,initial_demo,demo
1,VNLI_DEMO_0002,Giáo viên yêu cầu học sinh viết đoạn văn khoản...,Học sinh sẽ viết về một chủ đề xã hội.,ENT,nghi_luan,12,Ngu_van,SRC0001,author_owned,easy,no,author_v1,initial_demo,demo
2,VNLI_DEMO_0003,Văn bản cho biết tác giả sử dụng ngôi kể thứ n...,Người kể chuyện xưng tôi.,ENT,doc_hieu,11,Ngu_van,SRC0001,author_owned,easy,no,author_v1,initial_demo,demo
3,VNLI_DEMO_0004,Tài liệu ôn tập nhấn mạnh rằng học sinh không ...,Học tủ là cách học không được khuyến khích.,ENT,ky_nang_viet,12,Ngu_van,SRC0001,author_owned,easy,yes,author_v1,initial_demo,demo
4,VNLI_DEMO_0005,Đoạn văn mẫu đặt câu chủ đề ở câu đầu tiên.,Người đọc có thể nhận ra ý chính ngay từ đầu đ...,ENT,doc_hieu,10,Ngu_van,SRC0001,author_owned,easy,no,author_v1,initial_demo,demo


In [5]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nPhân bố nhãn train:")
print(train_df["label"].value_counts())

print("\nPhân bố nhãn validation:")
print(validation_df["label"].value_counts())

print("\nPhân bố nhãn test:")
print(test_df["label"].value_counts())

Train: (24, 14)
Validation: (3, 14)
Test: (3, 14)

Phân bố nhãn train:
label
CON    8
NEU    8
ENT    8
Name: count, dtype: int64

Phân bố nhãn validation:
label
ENT    1
CON    1
NEU    1
Name: count, dtype: int64

Phân bố nhãn test:
label
ENT    1
NEU    1
CON    1
Name: count, dtype: int64


In [ ]:
train_df = train_df.sort_values("id").reset_index(drop=True)
validation_df = validation_df.sort_values("id").reset_index(drop=True)
test_df = test_df.sort_values("id").reset_index(drop=True)
df = df.sort_values("id").reset_index(drop=True)

train_df.to_csv(WORKDIR / "train.csv", index=False)
validation_df.to_csv(WORKDIR / "validation.csv", index=False)
test_df.to_csv(WORKDIR / "test.csv", index=False)
df.to_csv(WORKDIR / "full_demo.csv", index=False)

schema_df.to_csv(META_DIR / "schema_vnli_educorpus.csv", index=False)
sources_df.to_csv(META_DIR / "sources_tracker.csv", index=False)

print("Đã lưu các file vào:", WORKDIR)
print(list(p.name for p in WORKDIR.iterdir()))
print("metadata:", list(p.name for p in META_DIR.iterdir()))

In [ ]:
HF_USERNAME = "ictkhangv"
HF_DATASET_REPO = f"{HF_USERNAME}/vnli-educorpus"

dataset_card = f"""---
language:
- vi
pretty_name: VNLI-EduCorpus
license: other
task_categories:
- text-classification
tags:
- vietnamese
- education
- natural-language-inference
- text
size_categories:
- n<1K
---

# VNLI-EduCorpus (v0.1-demo)

## Mô tả ngắn
VNLI-EduCorpus là bộ dữ liệu Suy luận ngôn ngữ tự nhiên (Natural Language Inference - suy luận ngôn ngữ tự nhiên) cho tiếng Việt trong lĩnh vực giáo dục.

Phiên bản hiện tại là **v0.1-demo**, dùng để kiểm tra hạ tầng dự án, cấu trúc dữ liệu, workflow upload và dataset card trên Hugging Face Hub.

## Thành phần dữ liệu
- `train.csv`
- `validation.csv`
- `test.csv`
- `full_demo.csv`
- `metadata/schema_vnli_educorpus.csv`
- `metadata/sources_tracker.csv`

## Nhãn
- `ENT`: Entailment (kéo theo)
- `CON`: Contradiction (mâu thuẫn)
- `NEU`: Neutral (trung tính)

## Phạm vi hiện tại
- Ngôn ngữ: tiếng Việt
- Miền: giáo dục
- Ưu tiên: Ngữ văn THPT

## Nguồn dữ liệu
Bản v0.1-demo hiện là dữ liệu demo tự biên soạn hoặc diễn giải hợp lệ để kiểm tra hạ tầng quy trình.

## Cảnh báo sử dụng
- Đây chưa phải bản phát hành chính thức cho đánh giá học thuật cuối cùng.
- Split hiện tại chỉ nhằm mục đích kiểm tra pipeline.
- Mọi dữ liệu do LLM hỗ trợ trong các phiên bản sau đều phải qua human review.

## Repo dự kiến
https://huggingface.co/datasets/{HF_DATASET_REPO}
"""

with open(WORKDIR / "README.md", "w", encoding="utf-8") as f:
    f.write(dataset_card)

print("Đã tạo README.md")
print((WORKDIR / "README.md").read_text(encoding="utf-8")[:800])

In [6]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
from huggingface_hub import HfApi

HF_USERNAME = "ictkhangv"
HF_REPO_ID = f"{HF_USERNAME}/vnli-educorpus"

api = HfApi()

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    private=True,
    exist_ok=True
)

api.upload_folder(
    folder_path=str(WORKDIR),
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    commit_message="Upload VNLI-EduCorpus v0.1-demo"
)

print("Upload xong.")
print(f"Repo: https://huggingface.co/datasets/{HF_REPO_ID}")

No files have been modified since last commit. Skipping to prevent empty commit.


Upload xong.
Repo: https://huggingface.co/datasets/ictkhangv/vnli-educorpus


In [ ]:
for name in ["train.csv", "validation.csv", "test.csv"]:
    temp = pd.read_csv(WORKDIR / name)
    print(f"\n{name}:")
    print(temp.shape)
    print(temp["label"].value_counts())
    display(temp.head(3))